# DAR Training Pipeline (Multi-Level Circle Loss)

Trains the LongCLIP MLP adapters using the augmented dataset generated by `data_augmentation.ipynb`.
This uses the **Multi-Level Circle Loss** to dynamically push away hard negatives across original and augmented modalities.

In [7]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from PIL import Image
import open_clip
from tqdm.auto import tqdm
import numpy as np
import os
from pathlib import Path
from lightning import Trainer, seed_everything
from lightning.pytorch import LightningModule
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger

device = 'cuda' if torch.cuda.is_available() else 'cpu'
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'paper_experiment' else Path.cwd()

CHECKPOINT_DIR = Path.cwd() / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / 'last.ckpt'

DATASET_PATH = Path("paired_dataset_dar.json")
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"{DATASET_PATH} not found. Please run data_augmentation.ipynb first.")

REPORTS_DIR = Path.cwd() / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

print(f'Using device: {device}')

Using device: cuda


## 1. Load LongCLIP Model & Adapters

In [8]:
MODEL_NAME = 'ViT-L-14'
PRETRAINED = 'openai'
CONTEXT_LENGTH = 248

def load_longclip(model_name, pretrained, context_length):
    try:
        model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained='longclip')
    except Exception:
        model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    
    if hasattr(model, 'positional_embedding') and model.positional_embedding.shape[0] != context_length:
        pe = model.positional_embedding
        new_pe = torch.zeros(context_length, pe.shape[1], device=pe.device, dtype=pe.dtype)
        new_pe[:pe.shape[0]] = pe
        new_pe[pe.shape[0]:] = pe[-1]
        model.positional_embedding = torch.nn.Parameter(new_pe)
        
    if hasattr(model, 'attn_mask') and model.attn_mask is not None and model.attn_mask.shape[0] != context_length:
        mask = torch.empty(context_length, context_length, device=model.attn_mask.device)
        mask.fill_(float("-inf"))
        mask.triu_(1)
        model.attn_mask = mask
        
    if hasattr(model, 'context_length'):
        model.context_length = context_length
    return model, preprocess

model, preprocess = load_longclip(MODEL_NAME, PRETRAINED, CONTEXT_LENGTH)
model = model.to(device)

def tokenize(texts):
    try:
        return open_clip.tokenize(texts, context_length=CONTEXT_LENGTH)
    except TypeError:
        tokenizer = open_clip.get_tokenizer(MODEL_NAME)
        return tokenizer(texts)

# Freeze base model
for param in model.parameters():
    param.requires_grad = False

EMBED_DIM = 768
ADAPTER_BOTTLENECK = 64

class Adapter(nn.Module):
    def __init__(self, dim=768, bottleneck=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, bottleneck),
            nn.ReLU(),
            nn.Linear(bottleneck, dim)
        )
        nn.init.zeros_(self.net[2].weight)
        nn.init.zeros_(self.net[2].bias)

    def forward(self, x):
        return x + self.net(x)

image_adapter = Adapter(EMBED_DIM, ADAPTER_BOTTLENECK).to(device)
text_adapter = Adapter(EMBED_DIM, ADAPTER_BOTTLENECK).to(device)

ERROR:root:Pretrained tag or path (longclip) for 'ViT-L-14' not found. Available tags: ['openai', 'laion400m_e31', 'laion400m_e32', 'laion2b_s32b_b82k', 'datacomp_xl_s13b_b90k', 'commonpool_xl_clip_s13b_b90k', 'commonpool_xl_laion_s13b_b90k', 'commonpool_xl_s13b_b90k', 'metaclip_400m', 'metaclip_fullcc', 'dfn2b', 'dfn2b_s39b']


## 2. Dataset Setup

In [9]:
class DARDataset(Dataset):
    def __init__(self, json_path, preprocess, tokenize, project_root):
        with open(json_path, 'r') as f:
            self.data = json.load(f)
        self.preprocess = preprocess
        self.tokenize = tokenize
        self.project_root = Path(project_root)

    def _resolve_path(self, path):
        p = Path(path)
        p_str = str(path).replace('../../', '')
        p = Path(p_str)
        return p if p.is_absolute() else (self.project_root / p).resolve()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        orig_img_path = self._resolve_path(item['image_path'])
        aug_img_path = self._resolve_path(item['aug_image_path'])
        
        orig_img = Image.open(orig_img_path).convert('RGB')
        aug_img = Image.open(aug_img_path).convert('RGB')
        
        orig_img_tensor = self.preprocess(orig_img)
        aug_img_tensor = self.preprocess(aug_img)
        
        # Original Text
        title = item.get('recipe_title', '')
        ingredients = item.get('ingredients', '')
        orig_text = f"Title: {title}\nIngredients: {ingredients}"
        
        # Augmented Text (LLM)
        aug_text = item.get('aug_visual_text', '')
        
        orig_text_tensor = self.tokenize([orig_text])[0]
        aug_text_tensor = self.tokenize([aug_text])[0]
        
        return orig_img_tensor, orig_text_tensor, aug_img_tensor, aug_text_tensor

dataset = DARDataset(DATASET_PATH, preprocess, tokenize, PROJECT_ROOT)
print(f"Dataset size: {len(dataset)} pairs")

BATCH_SIZE = 128
NUM_EPOCHS = 30
LR = 1e-4
NUM_WORKERS = 0  # Changed to 0 to avoid Jupyter CUDA fork deadlock

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device=='cuda'))

Dataset size: 5 pairs


## 3. Multi-Level Circle Loss

In [10]:
def circle_loss_sim(sim, margin=0.25, gamma=256):
    """
    Circle Loss on a similarity matrix.
    sim: NxN cosine similarity matrix.
    """
    # Dynamic weights
    alpha_p = torch.clamp_min(-sim.diag() + 1 + margin, min=0.0).detach()
    alpha_n = torch.clamp_min(sim + margin, min=0.0).detach()
    
    mask = torch.eye(sim.size(0), device=sim.device).bool()
    alpha_n = alpha_n.masked_fill(mask, 0.0)
    
    # Negatives
    logit_n = alpha_n * (sim - 0) * gamma
    logit_n = logit_n.masked_fill(mask, -1e4)
    logsumexp_n = torch.logsumexp(logit_n, dim=1)
    
    # Positives
    logit_p = -alpha_p * (sim.diag() - 1) * gamma
    
    loss = F.softplus(logsumexp_n + logit_p).mean()
    return loss

def multi_level_circle_loss(img_emb, txt_emb, aug_img_emb, aug_txt_emb, margin=0.25, gamma=80):
    # Normalize all embeddings
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)
    aug_img_emb = F.normalize(aug_img_emb, dim=-1)
    aug_txt_emb = F.normalize(aug_txt_emb, dim=-1)
    
    # Level 1: Original Image <-> Original Text
    sim_1 = img_emb @ txt_emb.T
    loss_1 = circle_loss_sim(sim_1, margin, gamma)
    
    # Level 2: Augmented Image <-> Augmented Text
    sim_2 = aug_img_emb @ aug_txt_emb.T
    loss_2 = circle_loss_sim(sim_2, margin, gamma)
    
    # Level 3: Original Image <-> Augmented Text
    sim_3 = img_emb @ aug_txt_emb.T
    loss_3 = circle_loss_sim(sim_3, margin, gamma)
    
    # Level 4: Augmented Image <-> Original Text
    sim_4 = aug_img_emb @ txt_emb.T
    loss_4 = circle_loss_sim(sim_4, margin, gamma)
    
    return (loss_1 + loss_2 + loss_3 + loss_4) / 4.0

## 4. Lightning Training Loop

In [11]:
seed_everything(42, workers=True)

class LitDAR(LightningModule):
    def __init__(self, model, image_adapter, text_adapter, lr):
        super().__init__()
        self.model = model
        self.image_adapter = image_adapter
        self.text_adapter = text_adapter
        self.lr = lr

    def on_train_start(self):
        self.model.eval() # Keep base frozen

    def training_step(self, batch, batch_idx):
        orig_img, orig_txt, aug_img, aug_txt = [b.to(self.device) for b in batch]
        
        with torch.no_grad():
            orig_img_feats = self.model.encode_image(orig_img).float()
            aug_img_feats = self.model.encode_image(aug_img).float()
            
            orig_txt_feats = self.model.encode_text(orig_txt).float()
            aug_txt_feats = self.model.encode_text(aug_txt).float()
            
        orig_img_emb = self.image_adapter(orig_img_feats)
        aug_img_emb = self.image_adapter(aug_img_feats)
        
        orig_txt_emb = self.text_adapter(orig_txt_feats)
        aug_txt_emb = self.text_adapter(aug_txt_feats)
        
        loss = multi_level_circle_loss(orig_img_emb, orig_txt_emb, aug_img_emb, aug_txt_emb)
        
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        params = list(self.image_adapter.parameters()) + list(self.text_adapter.parameters())
        optimizer = torch.optim.AdamW(params, lr=self.lr, weight_decay=0.01)
        scheduler = OneCycleLR(
            optimizer,
            max_lr=self.lr,
            total_steps=self.trainer.estimated_stepping_batches,
            pct_start=0.1,
            anneal_strategy='cos'
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}
        }

lit_model = LitDAR(model, image_adapter, text_adapter, LR)

checkpoint_callback = ModelCheckpoint(
    dirpath=str(CHECKPOINT_DIR),
    filename='dar_best_model',
    monitor='train_loss_epoch',
    mode='min',
    save_top_k=1,
    save_last=True
)
early_stopping = EarlyStopping(monitor='train_loss_epoch', patience=5, mode='min')
logger = CSVLogger(save_dir=str(REPORTS_DIR), name='dar_lightning_logs')

trainer = Trainer(
    max_epochs=NUM_EPOCHS,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    precision='16-mixed' if torch.cuda.is_available() else 32,
    callbacks=[checkpoint_callback, early_stopping],
    logger=logger,
    default_root_dir=str(CHECKPOINT_DIR)
)

ckpt_path = str(LAST_CHECKPOINT_PATH) if LAST_CHECKPOINT_PATH.exists() else None
print("Starting DAR Training...")
trainer.fit(lit_model, train_dataloaders=train_loader, ckpt_path=ckpt_path)
print("Training Complete!")

Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


Starting DAR Training...


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ CLIP    │  427 M │ train │     0 │
│ 1 │ image_adapter │ Adapter │ 99.1 K │ train │     0 │
│ 2 │ text_adapter  │ Adapter │ 99.1 K │ train │     0 │
└───┴───────────────┴─────────┴────────┴───────┴───────┘

Trainable params: 198 K                                                                                            
Non-trainable params: 427 M                                                                                        
Total params: 427 M                                                                                                
Total estimated model params size (MB): 1,711.784                                                                  
Modules in train mode: 418                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

RuntimeError: value cannot be converted to type c10::Half without overflow